# Experiment 10: where in its history does TiRex-2 look?

**Method (occlusion):** hide one hour of the history at a time (set it to "missing", which TiRex-2 accepts), run
the forecast again and measure how much the forecast changes. The model is deterministic, so every change is
caused by the hidden hour. Big change = the model relied on that hour.

**Cases:** 40 normal-day cases (Experiment 2) and 40 cold-front cases (Experiment 3, forecast starts 6 h before
the drop). History: **4 weeks** (672 h), temperature only, 24 h forecast. 672 hidden hours × 80 cases.

**Effect** = absolute change of the forecast median (°C) for each forecast hour.

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from tirex2 import TimeseriesType, load_model

DATA = Path("/workspace/data/austria_15km")
FIG = Path("/workspace/phase1-model-behavior/pair-1/experiments/figures/exp10")
FIG.mkdir(parents=True, exist_ok=True)
RES = Path("/workspace/data/notebooks/results")
L, H, N = 672, 24, 40
RESULTS = RES / f"exp10_occlusion_{N}x2.npz"
RERUN = False
QUICK = bool(os.environ.get("QUICK"))
if QUICK:
    N, FIG, RESULTS = 1, Path("/tmp"), Path("/tmp/exp10_test.npz")

df = pd.read_parquet(DATA / "grid_long.parquet", columns=["point_id", "time", "T2M"])
Tw = df.pivot(index="time", columns="point_id", values="T2M").interpolate(limit=3, limit_area="inside")
times, T = Tw.index, Tw.to_numpy(np.float32)
col = {p: i for i, p in enumerate(Tw.columns)}

rng = np.random.default_rng(0)
z2 = np.load(RES / "exp2_forecasts_50x48.npz")
k2 = rng.choice(len(z2["point_id"]), N, replace=False)
z3 = np.load(RES / "exp3_forecasts_30perfront.npz")
ok = (z3["truth"][:, 6] - z3["truth"][:, 6:13].min(1)) >= 2
k3 = rng.choice(np.where(ok)[0], N, replace=False)
cases = pd.concat([
    pd.DataFrame({"group": "normal day", "point_id": z2["point_id"][k2], "start": z2["start"][k2]}),
    pd.DataFrame({"group": "cold front", "point_id": z3["point_id"][k3], "start": z3["start"][k3]})], ignore_index=True)
cases["o"] = [times.get_loc(pd.Timestamp(s)) for s in cases.start]
print(cases.group.value_counts().to_dict(), f"-> {len(cases) * (L + 1)} forecasts")

{'normal day': 40, 'cold front': 40} -> 53840 forecasts


In [2]:
model = load_model("NX-AI/TiRex-2", device="cpu")
if RESULTS.exists() and not RERUN and not QUICK:
    eff = np.load(RESULTS)["effect"].astype(np.float32)
    print(f"loaded saved results from {RESULTS.name}")
else:
    eff = np.zeros((len(cases), L, H), np.float32)  # (case, hidden hour: 1 = last known hour ... 672, forecast hour)
    t0 = time.time()
    for i, (p, o) in enumerate(zip(cases.point_id, cases.o)):
        h = T[o - L : o, col[p]]
        xs = np.repeat(h[None], L + 1, axis=0)
        for lag in range(1, L + 1):
            xs[lag, L - lag] = np.nan
        out = model.forecast([TimeseriesType(torch.from_numpy(x[None].copy()), None, None) for x in xs],
                             prediction_length=H, output_type="numpy", batch_size=128)
        med = np.stack([q[0][4] for q in out])
        eff[i] = np.abs(med[1:] - med[0])
        if i % 10 == 0:
            print(f"case {i + 1}/{len(cases)}: {time.time() - t0:.0f} s")
    np.savez_compressed(RESULTS, effect=eff.astype(np.float16), group=cases.group.to_numpy().astype("U"),
                        point_id=cases.point_id.to_numpy(), start=cases.start.astype(str).to_numpy().astype("U"))
G = {g: eff[(cases.group == g).to_numpy()].mean(axis=0) for g in ["normal day", "cold front"]}  # (lag, forecast hour)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

loaded saved results from exp10_occlusion_40x2.npz


## Which hidden hour changes which forecast hour?

y = hidden hour (hours before the forecast start, last week only), x = forecast hour. Dotted lines mark hidden
hours at the **same hour of day** as the forecast hour, 1 to 7 days earlier. If the model copies the daily
rhythm, the bright cells line up along these diagonals. Colours are capped at the 98th percentile so that the
very last hours do not hide everything else.

In [3]:
LAGS = 168
zmax = np.percentile(np.stack([G[g][:LAGS] for g in G]), 98)
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04, subplot_titles=list(G))
for c, g in enumerate(G, start=1):
    fig.add_trace(go.Heatmap(z=G[g][:LAGS], x=np.arange(1, H + 1), y=np.arange(1, LAGS + 1), colorscale="Inferno", zmin=0, zmax=zmax,
                             showscale=c == 2, colorbar={"title": "change (°C)"},
                             hovertemplate="hide hour -%{y} h, forecast hour %{x}: %{z:.3f} °C<extra>" + g + "</extra>"), row=1, col=c)
    for d in range(1, 8):  # same hour of day: hidden hour at o - lag, forecast hour at o + x - 1 -> lag = 24 d - x + 1
        x = np.arange(1, H + 1)
        fig.add_trace(go.Scatter(x=x, y=24 * d - x + 1, mode="lines", line={"color": "rgba(120,200,255,0.6)", "width": 1, "dash": "dot"},
                                 hoverinfo="skip", showlegend=False), row=1, col=c)
    fig.update_xaxes(title_text="forecast hour", dtick=3, range=[0.5, H + 0.5], row=1, col=c)
fig.update_yaxes(title_text="hidden hour (hours before the forecast start)", autorange="reversed", range=[LAGS, 0.5], dtick=24, row=1, col=1)
fig.update_layout(height=720, template="plotly_dark",
                  title="Where does TiRex-2 look in the last week?<br><sup>mean change of the forecast when one hour of the history is hidden; "
                        f"{N} cases per group, 4-week history; dotted = same hour of day 1-7 days earlier</sup>")
fig.write_html(FIG / "exp10_lag_vs_forecast_hour.html", include_plotlyjs="cdn")
fig.show()

## The whole 4 weeks, folded by day

Each cell = one hidden hour, averaged over the 24 forecast hours. Rows = days back, columns = hours before the
forecast start within that day (column 1 = the hour just before the forecast start, 24 hours earlier, and so on).
Right: total effect per day back.

In [4]:
fold = {g: G[g].mean(axis=1).reshape(L // 24, 24) for g in G}  # (days back, hour within day)
zmax = np.percentile(np.stack(list(fold.values())), 98)
fig = make_subplots(rows=1, cols=3, column_widths=[0.37, 0.37, 0.26], horizontal_spacing=0.06,
                    subplot_titles=[*G, "Total per day back"])
for c, g in enumerate(G, start=1):
    fig.add_trace(go.Heatmap(z=fold[g], x=np.arange(1, 25), y=np.arange(1, L // 24 + 1), colorscale="Inferno", zmin=0, zmax=zmax,
                             showscale=c == 2, colorbar={"title": {"text": "change of the forecast (°C)", "side": "top"}, "orientation": "h",
                                                             "x": 0.37, "xanchor": "center", "y": -0.2, "yanchor": "top", "len": 0.5, "thickness": 12},
                             hovertemplate="day -%{y}, hour %{x} of that day: %{z:.3f} °C<extra>" + g + "</extra>"), row=1, col=c)
    fig.update_xaxes(title_text="hour within the day (1 = closest)", dtick=6, row=1, col=c)
for g, colr in [("normal day", "#7f7f7f"), ("cold front", "#d62728")]:
    fig.add_trace(go.Scatter(x=fold[g].sum(axis=1), y=np.arange(1, L // 24 + 1), mode="lines+markers", name=g,
                             line={"color": colr, "width": 2.5}, hovertemplate="day -%{y}: %{x:.2f} °C<extra>" + g + "</extra>"), row=1, col=3)
fig.update_yaxes(title_text="days before the forecast start", autorange="reversed", dtick=7, row=1, col=1)
fig.update_yaxes(autorange="reversed", dtick=7, row=1, col=2)
fig.update_yaxes(autorange="reversed", dtick=7, row=1, col=3)
fig.update_xaxes(title_text="summed change (°C)", row=1, col=3)
fig.update_layout(height=720, template="plotly_dark", margin={"b": 150},
                  legend={"orientation": "h", "x": 1, "xanchor": "right", "y": -0.2, "yanchor": "top"},
                  title="Which days of the history matter?<br><sup>change of the forecast (mean over the 24 forecast hours) "
                        "when one hour is hidden, folded into days</sup>")
fig.write_html(FIG / "exp10_days_folded.html", include_plotlyjs="cdn")
fig.show()
for g in G:
    d = fold[g].sum(axis=1)
    print(f"{g}: last day {d[0] / d.sum():.0%} of the total effect, last week {d[:7].sum() / d.sum():.0%}")

normal day: last day 12% of the total effect, last week 35%
cold front: last day 11% of the total effect, last week 35%


## Reusing the results

```python
z = np.load("/workspace/data/notebooks/results/exp10_occlusion_40x2.npz")
z["effect"]   # (cases, hidden hour 1-672 before the start, forecast hour 1-24): |change of the median|, °C
z["group"], z["point_id"], z["start"]   # one entry per case ("normal day" / "cold front")
```